# COL-02 — fresh Colab CPU sweep acceptance
Upload the locally verified handoff ZIP. No Drive mount, credentials or Docker. Run all once in a new CPU session. Preserve failures, download output and notebook with outputs; independently read back before Done.

In [ ]:
# External SHA is supplied by the LOCAL receipt, never by the archive itself.
import hashlib, json, os, re, stat, subprocess, sys, uuid, zipfile
from pathlib import Path, PurePosixPath
assert (3, 11) <= sys.version_info[:2] <= (3, 13), sys.version
ARCHIVE = Path(input("Absolute handoff ZIP path: ").strip())
TRUSTED_SHA = input("SHA256 from local handoff receipt: ").strip()
RECIPIENT = input("Operator (HoaiTam / rosy179 / ThanhTris): ").strip()
assert RECIPIENT in {"HoaiTam", "rosy179", "ThanhTris"}
assert re.fullmatch("[a-f0-9]{64}", TRUSTED_SHA)
assert hashlib.sha256(ARCHIVE.read_bytes()).hexdigest() == TRUSTED_SHA
SESSION = Path("/content/ie212-col02") / uuid.uuid4().hex
INPUT = SESSION / "input"
INPUT.mkdir(parents=True, exist_ok=False)
with zipfile.ZipFile(ARCHIVE) as z:
    entries = z.infolist()
    assert 0 < len(entries) <= 128 and len(entries) == len({e.filename for e in entries})
    assert sum(e.file_size for e in entries) <= 8 * 1024**2
    for e in entries:
        p = PurePosixPath(e.filename)
        assert not p.is_absolute() and ".." not in p.parts and "\\" not in e.filename
        assert str(p) == e.filename and not re.match("^[A-Za-z]:", e.filename)
        assert not e.is_dir() and not stat.S_ISLNK(e.external_attr >> 16) and not e.flag_bits & 1
        target = INPUT / e.filename
        target.parent.mkdir(parents=True, exist_ok=True)
        with target.open("xb") as f: f.write(z.read(e))
# Import trusted stdlib-only helper after the externally pinned ZIP is verified.
sys.dont_write_bytecode = True
sys.path.insert(0, str(INPUT / "code/notebooks"))
import colab_runtime as cr
m = cr.load(INPUT / "handoff.json")
assert set(m["files"]) == {p.relative_to(INPUT).as_posix() for p in INPUT.rglob("*") if p.is_file()} - {"handoff.json"}
assert all(cr.sha(INPUT / n) == h for n, h in m["files"].items())
print("PINNED_COL02_HANDOFF_BYTES_VERIFIED", TRUSTED_SHA)


In [ ]:
# Isolated CPU model environment; never mutate Colab host site-packages.
def checked(command):
    result = subprocess.run(command, capture_output=True, text=True, timeout=600)
    print(result.stdout)
    if result.stderr: print(result.stderr)
    result.check_returncode()
    return result

VENV = SESSION / "venv"
checked([sys.executable, "-m", "venv", "--without-pip", str(VENV)])
PYTHON = VENV / "bin/python"
checked([sys.executable, "-m", "pip", "--isolated", "--python", str(PYTHON), "install", "--disable-pip-version-check", "-r", str(INPUT / "code/notebooks/requirements-colab-lock.txt")])
checked([str(PYTHON), "-m", "pip", "check"])
print("ISOLATED_COL02_CPU_READY", sys.version.split()[0])


In [ ]:
WORK = SESSION / "work"
checked([str(PYTHON), "-B", str(INPUT / "code/scripts/col02-colab.py"), "run", "--handoff-root", str(INPUT), "--output", str(WORK), "--recipient", RECIPIENT])
print("COLAB_COL02_SWEEP_AND_PARITY_PASSED")


In [ ]:
# Export only reviewed result/metadata allowlist, no input/venv/token/cache.
EXPORT = SESSION / "export"
OUTPUT_ZIP = SESSION / "col02-colab-output.zip"
program = "import sys; from pathlib import Path; sys.path.insert(0,sys.argv[1]); import col02_runtime as c; c.export_output(Path(sys.argv[2]),Path(sys.argv[3])); print(c.archive_output(Path(sys.argv[3]),Path(sys.argv[4])))"
checked([str(PYTHON), "-B", "-c", program, str(INPUT / "code/notebooks"), str(WORK), str(EXPORT), str(OUTPUT_ZIP)])
print("COL02_OUTPUT_SHA256", hashlib.sha256(OUTPUT_ZIP.read_bytes()).hexdigest())
from google.colab import files
files.download(str(OUTPUT_ZIP))
